RAG BASICS

In [ ]:
git add Rag.ipynb; git commit -m "Add Rag notebook"; git push

In [ ]:
git add Rag.ipynb; git commit -m "Add Rag notebook"; git push

In [1]:
import langchain
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts.chat import PromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
google_api_key=os.getenv("GOOGLE_API_KEY")

google_llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    api_key=google_api_key,
    max_tokens = 50
)

In [2]:
#google_llm.invoke("Which is the last film of Vijay Thalapathy?")
#print(google_llm.invoke("What is your knowledge cutoff date?"))

1. Loading the document

In [3]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("Jn.txt",encoding="utf8")
documents =loader.load()
print(documents)

[Document(metadata={'source': 'Jn.txt'}, page_content='Jana Nayagan (Tamil pronunciation: [dʑanaː n̪aːjagan]; transl.\u2009People\'s Hero) is a 2026 Indian Tamil-language political action drama film directed by H. Vinoth and produced by Venkat K. Narayana under KVN Productions. The film stars C. Joseph Vijay, Bobby Deol, Pooja Hegde, and Mamitha Baiju in the lead role alongside Nassar, Prakash Raj, Gautham Vasudev Menon, Narain, and Priyamani. A partial remake of the 2023 Telugu film Bhagavanth Kesari, it follows a former Tamil Nadu Police Inspector who raises the daughter of a deceased friend and encourages her to overcome childhood trauma and pursue a career in the Indian Army. His efforts draw him into a confrontation with corrupt individuals, whose actions threaten democratic institutions.\n\nThe film was officially announced in September 2024 under the tentative title Thalapathy 69, as it was Vijay\'s 69th film as a lead actor, and the official title was announced in January 2025.

C:\Users\gokul\AppData\Local\Temp\ipykernel_6920\2328327593.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


2. Spliting the document(Chunking)

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pprint import pprint

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap= 50
)
chunks = text_splitter.split_documents(documents)
print(chunks[11].page_content)


officers washing ashore. Scientist Khaleel and his research team discover that the attackers employed an advanced technology called Remote Automation. Before Khaleel's team is eliminated, he escapes with crucial documents detailing the project. By coincidence, he entrusts them to Viji, asking her to deliver them to the Chief Minister, who is attending an event at her college.


3. Embedding and storing it to vector store

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

chunk_texts = [document.page_content for document in chunks]

doc_vectors = embeddings.embed_documents(chunk_texts)

print(doc_vectors[0][1:30])

[0.008570099, 0.02362818, -0.061381858, -0.005566337, 0.020886967, -0.021456892, 0.0063276296, -0.0026529303, -0.014277006, -0.01747361, -0.0058751074, -0.008329873, 0.040927984, 0.12277844, -0.02097808, 0.010686417, 0.0047307466, 0.007666129, -0.0043132887, 0.00095414463, -0.038086876, 0.0119713815, 0.0038056255, -0.006311508, -0.019687768, 0.023489662, 0.010829758, 0.02449969, -0.011723805]


4. Storing it to vector database

In [8]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks,embeddings)
result = vectorstore.similarity_search("Who is kayal? answer in short?")
print(result)

[Document(id='b418c5b8-8a22-4ec5-b8e5-4a4da1633198', metadata={'source': 'Jn.txt'}, page_content="for her increasingly difficult. Determined to fulfil her parents' dream, Vetri seeks the help of journalist Kayal to encourage Viji to join the Army despite her reluctance."), Document(id='72489cfb-1a29-4480-9660-943089419b58', metadata={'source': 'Jn.txt'}, page_content="officers washing ashore. Scientist Khaleel and his research team discover that the attackers employed an advanced technology called Remote Automation. Before Khaleel's team is eliminated, he escapes with crucial documents detailing the project. By coincidence, he entrusts them to Viji, asking her to deliver them to the Chief Minister, who is attending an event at her college."), Document(id='ef63ffeb-cb2d-4c2c-bb6c-9c40ed4f9cda', metadata={'source': 'Jn.txt'}, page_content="As Vetri investigates the conspiracy with Kayal, he discovers that Viji's neurosis originated from a traumatic childhood incident in which she witness

5.Making vectorestore into runnable 

In [9]:
retriever=vectorstore.as_retriever()
print(retriever.invoke("Who is Viji biological dad?"))

[Document(id='4c38a8ca-9b3a-4f04-9d8d-a6bf4523909f', metadata={'source': 'Jn.txt'}, page_content='Plot\nVetri is released from prison after attracting the respect of jailer Srikanth. Before his death, Srikanth entrusts Vetri with the care of his daughter, Vijayalakshmi "Viji". Honouring the wishes of Srikanth and his late wife, Vetri raises Viji with the dream of seeing her become an officer in the Indian Army.'), Document(id='8b0b9c9e-15cf-437a-b899-9d2768672c44', metadata={'source': 'Jn.txt'}, page_content="By 2026, Vetri has settled into a peaceful life as the owner of a supermarket. Viji is nearing the completion of her college education and is in a relationship with her classmate Karthik. She wishes to marry and lead a normal life instead of pursuing a military career. At the same time, she struggles with severe neurosis, making Vetri's ambitions for her increasingly difficult. Determined to"), Document(id='b418c5b8-8a22-4ec5-b8e5-4a4da1633198', metadata={'source': 'Jn.txt'}, page

6. Feed and get response from LLM

In [17]:
from langchain_core.runnables import RunnablePassthrough

# Create RAG chain using LCEL
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt_template = ChatPromptTemplate.from_template("""
        Use the following context to answer the question:
        Context: {context}
        Question: {question}

        Answer:""")


chain = {"context": retriever|format_docs, "question": RunnablePassthrough()} | prompt_template | google_llm | StrOutputParser()

print(chain.invoke("Who is Vetri kondan?"))

Based on the provided context, Vetri is a man who is released from prison after gaining the respect of jailer Srikanth. Before Srikanth's death, he entrusts Vetri with the care of his daughter
